# Maths-2: Linear programming — geometry, vertices and duality

## Objectives

At the end of this notebook you should be able to:

- write a dispatch problem as a **linear program** (LP) and draw its feasible region
- explain why an optimal solution sits at a **vertex**, and why solvers do not simply try all vertices
- read **dual values** (shadow prices) as the change in total cost per unit of a constraint — and check them by finite differences
- see why the dual of the demand constraint is the **electricity price** and the dual of an emissions cap is a **carbon price**
- build the **dual LP** explicitly and verify **strong duality** and **complementary slackness**
- recognise **degenerate** cases where the price is not unique

In M1.2 you met PyPSA's prices as "the dual of the nodal balance constraint" and saw that they equal the merit-order price. This lesson explains *why*, with the smallest problem that shows everything: two generators and a CO₂ cap. Module 03 builds on it with unit commitment, storage and Linopy.

In [1]:
import itertools
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pypsa
from scipy.optimize import linprog

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### A dispatch problem with a CO₂ cap

Two plants serve a demand of 1,000 MW for one hour, under a cap of 600 t of CO₂:

| Plant | Marginal cost | Capacity | Emissions |
|---|---|---|---|
| coal | 60 €/MWh | 600 MW | 0.90 t/MWh |
| gas | 90 €/MWh | 800 MW | 0.35 t/MWh |

As a linear program, with generation g₁, g₂ in MW:

**minimise** 60·g₁ + 90·g₂ (€/h)
**subject to** g₁ + g₂ = 1000 (demand), 0.9·g₁ + 0.35·g₂ ≤ 600 (CO₂ cap), 0 ≤ g₁ ≤ 600, 0 ≤ g₂ ≤ 800

Without the cap, coal would run flat out (600 MW, 540 t) and gas would supply the rest (400 MW, 140 t): 680 t — over the cap. So the cap will bind. Solve it with HiGHS through SciPy:

In [2]:
cost = np.array([60.0, 90.0])          # EUR/MWh
emis = np.array([0.90, 0.35])          # t/MWh
cap = np.array([600.0, 800.0])         # MW
DEMAND, CO2_CAP = 1000.0, 600.0        # MW, t


def dispatch(demand=DEMAND, co2_cap=CO2_CAP):
    """Solve the two-plant dispatch LP with HiGHS."""
    return linprog(cost, A_ub=[emis], b_ub=[co2_cap], A_eq=[[1.0, 1.0]], b_eq=[demand],
                   bounds=list(zip([0.0, 0.0], cap)), method="highs")


res = dispatch()
print(f"coal {res.x[0]:.1f} MW, gas {res.x[1]:.1f} MW, total cost {res.fun:,.0f} EUR/h, "
      f"emissions {emis @ res.x:.1f} t")

coal 454.5 MW, gas 545.5 MW, total cost 76,364 EUR/h, emissions 600.0 t


Coal is cut back to 454.5 MW so that emissions land exactly on the cap, and gas covers the rest. Now the geometry of *why* this is the answer.

### The feasible region and its vertices

To draw the problem in two dimensions, write the demand as **g₁ + g₂ ≥ 1000**. With positive costs the solver never produces more than needed, so the optimum is the same — but now every constraint is a half-plane, and their intersection is a **polygon**: the feasible region.

Its corners — **vertices** — are where two constraint lines cross. Find them all by brute force: intersect every pair of lines and keep the points that satisfy every constraint:

In [3]:
# every constraint as a row of A x <= b
A = np.array([[-1, 0], [0, -1], [1, 0], [0, 1], [0.90, 0.35], [-1, -1]], dtype=float)
b = np.array([0, 0, 600, 800, CO2_CAP, -DEMAND])
names = ["coal >= 0", "gas >= 0", "coal <= 600", "gas <= 800", "CO2 cap", "demand"]

vertices = []
for i, j in itertools.combinations(range(len(A)), 2):
    M = A[[i, j]]
    if abs(np.linalg.det(M)) < 1e-12:
        continue                                     # parallel lines never cross
    x = np.linalg.solve(M, b[[i, j]])
    if np.all(A @ x <= b + 1e-9):
        vertices.append({"coal_MW": x[0], "gas_MW": x[1], "cost_EUR": cost @ x,
                         "binding": f"{names[i]} & {names[j]}"})
vertices = pd.DataFrame(vertices).drop_duplicates(subset=["coal_MW", "gas_MW"]).sort_values("cost_EUR")
best = vertices.iloc[0]
assert np.allclose([best.coal_MW, best.gas_MW], res.x), "the cheapest vertex is the LP optimum"
vertices.round(1)

   coal_MW  gas_MW  cost_EUR               binding
2    454.5   545.5   76363.6      CO2 cap & demand
1    200.0   800.0   84000.0   gas <= 800 & demand
0    355.6   800.0   93333.3  gas <= 800 & CO2 cap

In [4]:
fig, ax = plt.subplots(figsize=(6, 5))
g1 = np.linspace(0, 700, 400)
ax.plot(g1, DEMAND - g1, label="demand: g1 + g2 = 1000")
ax.plot(g1, (CO2_CAP - 0.9 * g1) / 0.35, label="CO2 cap")
ax.axvline(600, color="grey", ls=":", label="coal capacity")
ax.axhline(800, color="grey", ls="--", label="gas capacity")
poly = vertices[["coal_MW", "gas_MW"]].to_numpy()
order = np.argsort(np.arctan2(poly[:, 1] - poly[:, 1].mean(), poly[:, 0] - poly[:, 0].mean()))
ax.fill(*poly[order].T, alpha=0.25, label="feasible region")
for level in [70_000, best.cost_EUR, 85_000]:
    ax.plot(g1, (level - 60 * g1) / 90, color="k", lw=0.6, alpha=0.5)
ax.plot(best.coal_MW, best.gas_MW, "ko", ms=8, label="optimum (a vertex)")
ax.set(xlim=(0, 700), ylim=(0, 1100), xlabel="coal generation g1 (MW)", ylabel="gas generation g2 (MW)",
       title="Feasible region and iso-cost lines")
ax.legend(fontsize=8, loc="upper right")
plt.show()

The region is a triangle with three vertices. The thin black lines are **iso-cost lines** (all points with the same total cost); they are parallel, and moving them towards lower cost, the last point of the region they touch is a vertex. That is the **fundamental theorem of linear programming**: if an LP has an optimum, one of the vertices is optimal.

So why not always enumerate vertices? Because their number explodes. With n variables and m constraints there can be up to C(m, n) candidate intersections:

In [5]:
from math import lgamma, log


def digits_of_binomial(m, n):
    """Number of decimal digits of C(m, n), computed with logarithms (the number itself is too big to print)."""
    log10_c = (lgamma(m + 1) - lgamma(n + 1) - lgamma(m - n + 1)) / log(10)
    return int(log10_c) + 1


for n_vars, n_cons in [(2, 6), (10, 30), (100, 300), (8760 * 5, 8760 * 12)]:
    print(f"{n_vars:>6} variables, {n_cons:>6} constraints: up to a number with {digits_of_binomial(n_cons, n_vars):>6} digits")

     2 variables,      6 constraints: up to a number with      2 digits
    10 variables,     30 constraints: up to a number with      8 digits
   100 variables,    300 constraints: up to a number with     82 digits
 43800 variables, 105120 constraints: up to a number with  31005 digits


A one-year hourly dispatch with five plants already has tens of thousands of variables, and the candidate count runs to tens of thousands of *digits* — so large that Python refuses even to print it, which is why the cell counts digits with logarithms. The **simplex method** walks from vertex to neighbouring vertex, always downhill, and usually needs only a small multiple of the number of constraints; **interior-point methods** cut through the middle of the region. HiGHS has both. You will open them up in Module 03.

### Duality: what a constraint is worth

Every constraint has a **dual value** (a *shadow price*): how much the optimal cost changes per unit of change in the constraint's right-hand side. Measure it directly by finite differences — re-solve with slightly more demand, or a slightly looser cap:

In [6]:
h = 1e-3
price_fd = (dispatch(DEMAND + h).fun - dispatch(DEMAND - h).fun) / (2 * h)
co2_fd = (dispatch(co2_cap=CO2_CAP + h).fun - dispatch(co2_cap=CO2_CAP - h).fun) / (2 * h)
print(f"one more MWh of demand costs {price_fd:.2f} EUR   (solver dual: {res.eqlin.marginals[0]:.2f})")
print(f"one more tonne of CO2 allowed saves {-co2_fd:.2f} EUR (solver dual: {res.ineqlin.marginals[0]:.2f})")
assert np.isclose(price_fd, res.eqlin.marginals[0]) and np.isclose(co2_fd, res.ineqlin.marginals[0])

one more MWh of demand costs 109.09 EUR   (solver dual: 109.09)
one more tonne of CO2 allowed saves 54.55 EUR (solver dual: -54.55)


The dual of the demand constraint is the **electricity price**: 109.09 €/MWh. The dual of the CO₂ cap is a **carbon price**: relaxing the cap by one tonne saves 54.55 € (the solver reports it as −54.55 because relaxing a "≤" constraint *lowers* the cost).

Look at the electricity price: it is **higher than both plants' costs** (60 and 90 €/MWh). Serving one more MWh within the cap cannot be done by gas alone — that would add 0.35 t — so coal must be cut back too. The cheapest way to serve one extra MWh is to raise gas by 18/11 ≈ 1.64 MWh and lower coal by 7/11 ≈ 0.64 MWh, which keeps emissions unchanged: (18 × 90 − 7 × 60) / 11 = 109.09 €.

### Checking by hand: the equal-marginal condition

Both plants run strictly between their limits, so for each the cost of one more MWh *including its carbon* must equal the price:

**λ = c_i + μ · e_i**  for both i  →  60 + 0.9 μ = 90 + 0.35 μ

Two equations, two unknowns:

In [7]:
mu = (cost[1] - cost[0]) / (emis[0] - emis[1])     # EUR per tonne
lam = cost[0] + mu * emis[0]                       # EUR per MWh
assert np.isclose(lam, res.eqlin.marginals[0]) and np.isclose(mu, -res.ineqlin.marginals[0])
print(f"carbon price mu = {mu:.2f} EUR/t, electricity price lambda = {lam:.2f} EUR/MWh")

carbon price mu = 54.55 EUR/t, electricity price lambda = 109.09 EUR/MWh


Pen-and-paper algebra, finite differences and the solver agree. The carbon price is exactly the one that makes the two plants equally attractive at the margin — an emissions-trading price works the same way.

### The dual problem

Every LP has a twin, its **dual**. For a primal *minimise cᵀx subject to A x ≤ b* the dual is *maximise −bᵀy subject to Aᵀy = −c, y ≥ 0* — one dual variable y per constraint, one dual constraint per primal variable. Build and solve it for the inequality form above:

In [8]:
dual = linprog(b, A_eq=A.T, b_eq=-cost, bounds=[(0, None)] * len(b), method="highs")
primal = linprog(cost, A_ub=A, b_ub=b, method="highs")
assert np.isclose(-dual.fun, primal.fun), "strong duality: the optimal values are equal"

slack = b - A @ primal.x
cs = pd.DataFrame({"constraint": names, "slack": slack, "dual_value": dual.x})
assert np.allclose(cs.slack * cs.dual_value, 0, atol=1e-6), "complementary slackness"
print(f"primal optimum {primal.fun:,.2f} EUR = dual optimum {-dual.fun:,.2f} EUR")
cs.round(3)

primal optimum 76,363.64 EUR = dual optimum 76,363.64 EUR


    constraint    slack  dual_value
0    coal >= 0  454.545       0.000
1     gas >= 0  545.455       0.000
2  coal <= 600  145.455       0.000
3   gas <= 800  254.545       0.000
4      CO2 cap    0.000      54.545
5       demand    0.000     109.091

Two facts every LP satisfies, here checked numerically:

- **Strong duality**: the primal minimum and the dual maximum are equal (76,363.64 €). The dual's value is "what the constraints are worth" — the price of demand times demand, minus the value of the scarce CO₂ allowance.
- **Complementary slackness**: each constraint either binds (slack 0) or has dual value 0. The capacity limits are not binding, so they are worth nothing; demand and the CO₂ cap bind, so they carry the prices.

In markets these duals are exactly the money flows: consumers pay λ per MWh, and a binding cap creates scarcity rent μ per tonne.

### The same LP in PyPSA

PyPSA builds this LP from components: the CO₂ cap is a `GlobalConstraint`. Its prices should match:

In [9]:
n = pypsa.Network()
n.set_snapshots([0])
n.add("Bus", "bus")
n.add("Carrier", "coal", co2_emissions=0.90)
n.add("Carrier", "gas", co2_emissions=0.35)
n.add("Generator", "coal", bus="bus", carrier="coal", p_nom=600, marginal_cost=60)
n.add("Generator", "gas", bus="bus", carrier="gas", p_nom=800, marginal_cost=90)
n.add("Load", "demand", bus="bus", p_set=DEMAND)
n.add("GlobalConstraint", "co2_cap", type="primary_energy", carrier_attribute="co2_emissions",
      sense="<=", constant=CO2_CAP)
n.optimize(solver_name="highs", log_to_console=False)

pypsa_price = n.buses_t.marginal_price.iloc[0, 0]
pypsa_mu = n.global_constraints.mu["co2_cap"]
assert np.isclose(pypsa_price, lam) and np.isclose(abs(pypsa_mu), mu) and np.isclose(n.objective, res.fun)
print(f"PyPSA: price {pypsa_price:.2f} EUR/MWh, CO2 constraint dual {pypsa_mu:.2f} EUR/t, cost {n.objective:,.0f} EUR")

PyPSA: price 109.09 EUR/MWh, CO2 constraint dual -54.55 EUR/t, cost 76,364 EUR


Identical. Note the **sign**: PyPSA, like SciPy, reports the CO₂ dual as negative. Sign conventions for duals differ between solvers and tools, so always check against a finite difference before interpreting one — a lesson that will save you time in Module 03.

### When the price is not unique: degeneracy

Remove the cap and set demand to exactly 600 MW — precisely coal's capacity. What is the price?

In [10]:
def no_cap(demand):
    return linprog(cost, A_eq=[[1.0, 1.0]], b_eq=[demand], bounds=list(zip([0.0, 0.0], cap)), method="highs")

left = no_cap(600).fun - no_cap(599).fun
right = no_cap(601).fun - no_cap(600).fun
print(f"cost of the last MWh before 600 MW: {left:.0f} EUR | of the next MWh after: {right:.0f} EUR")
print(f"the solver reports a price of {no_cap(600).eqlin.marginals[0]:.0f} EUR/MWh")

cost of the last MWh before 600 MW: 60 EUR | of the next MWh after: 90 EUR
the solver reports a price of 90 EUR/MWh


Below 600 MW, an extra MWh comes from coal (60 €); above it, from gas (90 €). At exactly 600 MW the cost curve has a **kink**, and *every* price between 60 and 90 €/MWh is a valid dual — the LP is **degenerate**. The solver returns one of them, and which one can depend on the solver or even its settings. Real power exchanges meet this case every day (a step in the merit order exactly at the demand), and their market rules decide which price applies. When a model's prices jump unexpectedly between nearly identical runs, degeneracy is the first suspect.

## Check your understanding

**Geometry**

1. Raise the CO₂ cap to 700 t. Which vertex becomes optimal, and which constraint stops binding? Predict the electricity price before computing it.
2. Why can an LP optimum never lie strictly inside the feasible region (unless the objective is flat)?

**Duality**

1. The carbon price is 54.55 €/t. An emissions-trading market prices CO₂ at 80 €/t instead, with no cap. How would you add that to the model, and which plant would run more?
2. In complementary slackness terms, why is the dual of the gas capacity limit zero here? When would it become positive, and what would that mean for the price?